# From Pilot to Payoff - 01: Setup & Data Preparation

## Problem, Motivation and Objectives

### Problem and Motivation

AI adoption is growing rapidly across countries and industries, but adoption itself does not guarantee measurable business value or responsible workforce outcomes. Some firms remain in experimentation or pilot stages, while others move into partial or full adoption and appear to realise stronger productivity, revenue, cost, innovation, and workforce outcomes.

This project studies the path from **pilot to payoff**: what country-level and firm-level factors are associated with advanced AI adoption, and whether advanced adoption is associated with business value and workforce outcomes.

### Objectives

1. Describe AI adoption maturity across countries, regions, industries, company sizes, and time.
2. Examine whether country-level digital readiness is associated with advanced AI adoption.
3. Identify which representative firm-level drivers from capability, investment, and governance/risk blocks are most strongly associated with advanced adoption after controlling for country/region context, industry, company size, and time.
4. Test whether advanced adoption is associated with measurable business outcomes.
5. Assess whether advanced adoption is associated with workforce reskilling, net job creation, and lower AI failure rates.

### Analytical Questions

| # | Analytical question | Main methods used |
|---|---|---|
| Q1 | How does AI adoption maturity vary across countries, regions, industries, company sizes, and time? | EDA, grouped summaries, visualisation |
| Q2 | Are country-level digital readiness factors associated with advanced AI adoption? | Correlation, VIF, PCA, OLS |
| Q3 | After controlling for country/region context, industry, company size, and time, which representative firm-level drivers from capability, investment, and governance/risk blocks are most strongly associated with advanced AI adoption after checking multicollinearity and model contribution? | Chi-square, Kruskal-Wallis, VIF, standardised logistic regression, nested model comparison |
| Q4 | Is advanced adoption associated with business payoff, and does payoff differ by national digital maturity? | Mann-Whitney/Kruskal-Wallis, OLS, adjusted R2, interaction terms |
| Q5 | Is advanced adoption associated with workforce outcomes and lower implementation risk? | OLS, latest-company robustness, Beta-Binomial Bayesian inference |


## Data Sources and Preparation

### Dataset Description

The analysis uses three files:

1. `ai_company_adoption.csv`: company-quarter level dataset with AI adoption, firm characteristics, governance, business outcomes, and workforce outcomes.
2. `country_ai_index.csv`: country-level digital readiness indicators used for Q2 and moderation analysis.
3. `ai_industry_summary.csv`: industry-level summary file used only as a background validation reference, not as the primary modelling data.

The key unit of observation in the main dataset is **firm-quarter**, not independent firms. This matters because repeated observations for the same company can inflate significance tests. The notebook therefore includes a latest-observation-per-company robustness check for selected analyses.


## Setup and Imports

This data-preparation module uses `pandas` and `numpy`. The analysis modules (02-07) add the rest of the course stack as needed: `scipy.stats`, `statsmodels`, `matplotlib`, `seaborn`, and `scikit-learn` (PCA / standardisation).


In [1]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd

In [2]:
company = pd.read_csv('ai_company_adoption.csv')
country_index = pd.read_csv('country_ai_index.csv')
industry_summary = pd.read_csv('ai_industry_summary.csv')

print('Dataset shapes')
print(f'company adoption: {company.shape[0]:,} rows x {company.shape[1]:,} columns')
print(f'country index:    {country_index.shape[0]:,} rows x {country_index.shape[1]:,} columns')
print(f'industry summary: {industry_summary.shape[0]:,} rows x {industry_summary.shape[1]:,} columns')

company.head(3)


Dataset shapes
company adoption: 150,000 rows x 43 columns
country index:    30 rows x 8 columns
industry summary: 9 rows x 8 columns


,response_id,company_id,survey_year,quarter,country,region,industry,company_size,num_employees,annual_revenue_usd_millions,...,productivity_change_percent,jobs_displaced,jobs_created,reskilled_employees,revenue_growth_percent,cost_reduction_percent,innovation_score,customer_satisfaction,survey_source,data_collection_method
0,1,COMP-00001,2023,Q1,Italy,Europe,Education,Startup,57,48.31,...,2.65,1,1,3,2.52,9.45,53,5.20,WEF Survey,API Scrape
1,2,COMP-00001,2023,Q2,Italy,Europe,Education,Startup,57,48.31,...,5.77,2,2,5,4.77,0.00,51,6.98,McKinsey Report,Phone Interview
2,3,COMP-00001,2023,Q3,Italy,Europe,Education,Startup,57,48.31,...,6.94,3,3,2,12.87,9.74,40,4.12,Internal Corporate Survey,Research Compilation


In [3]:
# Data quality checks: missing values, duplicates, and panel structure.
missing_summary = company.isna().sum().loc[lambda s: s > 0]
duplicate_rows = company.duplicated().sum()
company_counts = company.groupby('company_id').size()

quality_summary = pd.DataFrame({
    'metric': [
        'Rows in main dataset',
        'Columns in main dataset',
        'Unique companies',
        'Duplicate full rows',
        'Minimum observations per company',
        'Median observations per company',
        'Maximum observations per company',
        'Columns with missing values'
    ],
    'value': [
        f'{len(company):,}',
        f'{company.shape[1]:,}',
        f'{company.company_id.nunique():,}',
        f'{duplicate_rows:,}',
        int(company_counts.min()),
        int(company_counts.median()),
        int(company_counts.max()),
        int(len(missing_summary))
    ]
})

quality_summary


,metric,value
0,Rows in main dataset,"150,000"
1,Columns in main dataset,43
2,Unique companies,"10,000"
3,Duplicate full rows,0
4,Minimum observations per company,10
5,Median observations per company,15
6,Maximum observations per company,16
7,Columns with missing values,0


In [4]:
# Simple data dictionary for the variables used in the final analysis.
data_dictionary = pd.DataFrame([
    ('ai_adoption_stage', 'categorical', 'AI maturity stage: none, pilot, partial, full'),
    ('advanced_adoption', 'binary engineered', '1 if partial/full adoption, 0 if none/pilot'),
    ('country, region, industry, company_size', 'categorical', 'Segmentation and control variables'),
    ('num_employees, annual_revenue_usd_millions, company_age', 'numeric', 'Firm scale and age controls'),
    ('ai_training_hours, num_ai_tools_used, ai_projects_active, years_using_ai', 'numeric', 'Capability and experience driver block'),
    ('ai_budget_percentage, ai_investment_per_employee', 'numeric', 'Investment driver block'),
    ('regulatory_compliance_score, data_privacy_level, ai_ethics_committee, ai_risk_management_score', 'numeric/categorical', 'Governance and risk management driver block'),
    ('digital_maturity_index, internet_penetration, gdp_per_capita, ai_patent_filings_2024, ai_researchers_per_million', 'numeric', 'Country digital readiness indicators'),
    ('productivity_change_percent, revenue_growth_percent, cost_reduction_percent, innovation_score, customer_satisfaction', 'numeric', 'Business payoff outcomes'),
    ('jobs_created, jobs_displaced, reskilled_employees, ai_failure_rate', 'numeric', 'Workforce and implementation risk outcomes'),
], columns=['variable', 'type', 'description'])

data_dictionary


,variable,type,description
0,ai_adoption_stage,categorical,"AI maturity stage: none, pilot, partial, full"
1,advanced_adoption,binary engineered,"1 if partial/full adoption, 0 if none/pilot"
2,"country, region, industry, company_size",categorical,Segmentation and control variables
3,"num_employees, annual_revenue_usd_millions, co...",numeric,Firm scale and age controls
4,"ai_training_hours, num_ai_tools_used, ai_proje...",numeric,Capability and experience driver block
5,"ai_budget_percentage, ai_investment_per_employee",numeric,Investment driver block
6,"regulatory_compliance_score, data_privacy_leve...",numeric/categorical,Governance and risk management driver block
7,"digital_maturity_index, internet_penetration, ...",numeric,Country digital readiness indicators
8,"productivity_change_percent, revenue_growth_pe...",numeric,Business payoff outcomes
9,"jobs_created, jobs_displaced, reskilled_employ...",numeric,Workforce and implementation risk outcomes


In [5]:
# Feature engineering and merge.
quarter_map = {'Q1': 1, 'Q2': 2, 'Q3': 3, 'Q4': 4}
company = company.copy()
company['quarter_num'] = company['quarter'].map(quarter_map)
company['advanced_adoption'] = company['ai_adoption_stage'].isin(['partial', 'full']).astype(int)
company['net_jobs_created'] = company['jobs_created'] - company['jobs_displaced']
company['net_jobs_per_100_employees'] = company['net_jobs_created'] / company['num_employees'] * 100
company['reskilling_rate_per_100_employees'] = company['reskilled_employees'] / company['num_employees'] * 100

# Log transform for the skewed investment variable.
# 780 firms (0.52%) report zero AI investment; np.log1p collapses these to 0, far below the
# positive mass (median log ~10.7), creating an artificial left-skewed spike (skew -2.8) that
# distorts the standardised predictor feeding the Q3 logistic model. Instead, log the positive
# amounts (near-symmetric, skew -0.43) and set the rare zeros to the median log, so they sit at
# the centre rather than forming a distorting tail.
_pos_investment = company['ai_investment_per_employee'] > 0
_log_investment = np.log(company['ai_investment_per_employee'].where(_pos_investment))
company['log_ai_investment_per_employee'] = _log_investment.fillna(_log_investment.median())

country_model = country_index.drop(columns=['region'])
country_model['log_gdp_per_capita'] = np.log1p(country_model['gdp_per_capita'])
country_model['log_ai_patent_filings_2024'] = np.log1p(country_model['ai_patent_filings_2024'])

df = company.merge(country_model, on='country', how='left')

# Latest observation per company for robustness checks.
latest_obs = (df.sort_values(['company_id', 'survey_year', 'quarter_num'])
                .groupby('company_id', as_index=False)
                .tail(1)
                .reset_index(drop=True))

# Digital maturity tertiles for interaction models.
df['digital_maturity_tertile'] = pd.qcut(
    df['digital_maturity_index'], q=3, labels=['Low', 'Medium', 'High']
)
latest_obs['digital_maturity_tertile'] = pd.qcut(
    latest_obs['digital_maturity_index'], q=3, labels=['Low', 'Medium', 'High']
)

merge_check = df[['country', 'digital_maturity_index']].isna().mean().rename('missing_rate')
print(f'Merged modelling data: {df.shape[0]:,} rows x {df.shape[1]:,} columns')
print(f'Latest-company robustness data: {latest_obs.shape[0]:,} rows')
print('\nCountry-index missing check:')
print(merge_check)


Merged modelling data: 150,000 rows x 58 columns
Latest-company robustness data: 10,000 rows

Country-index missing check:
country                   0.0
digital_maturity_index    0.0
Name: missing_rate, dtype: float64


In [6]:
# Standardise numeric predictors used in regression models.
standardise_cols = [
    'ai_training_hours', 'num_ai_tools_used', 'ai_projects_active', 'years_using_ai',
    'ai_budget_percentage', 'log_ai_investment_per_employee',
    'regulatory_compliance_score', 'ai_risk_management_score'
]

for col in standardise_cols:
    mean = df[col].mean()
    std = df[col].std(ddof=0)
    df[f'z_{col}'] = (df[col] - mean) / std
    latest_obs[f'z_{col}'] = (latest_obs[col] - mean) / std

print('Created z-score variables for regression comparability.')


Created z-score variables for regression comparability.
